In [ ]:
# must be set before CUDA is initialised (required by torch.use_deterministic_algorithms)
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import argparse
import random  # required for set_seed2() reproducibility helper
import numpy as np
import pandas as pd
import os
import cv2
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data.dataset import Dataset
from torch.utils.data import DataLoader
import albumentations as A
import torchmetrics as tm

import sys 
sys.path.append('main-codes/Mamba-UNet-main/code')
from networks.vision_transformer import SwinUnet as ViT_seg
from networks.vision_mamba import MambaUnet as VIM_seg
# DataParallel fix: SS2D binds forward_core to the original module in __init__,
# so replicas on cuda:1/2 would use cuda:0 weights. Make it a class attribute instead.
from networks.mamba_sys import SS2D
_ss2d_init = SS2D.__init__
def _ss2d_init_patched(self, *a, **k):
    _ss2d_init(self, *a, **k)
    self.__dict__.pop("forward_core", None)
SS2D.__init__ = _ss2d_init_patched
SS2D.forward_core = SS2D.forward_corev0

sys.path.append("main-codes/modified_segmentation_models-main") 
import segmentation_models_pytorch as smp 

sys.path.append("main-codes/HybridModels")
from HybridSwinUNetC import HybridSwinUNetC
from HybridMambaUNetC import HybridMambaUNetC

from config import get_config

import copy
import yacs.config
torch.serialization.add_safe_globals([yacs.config.CfgNode])

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
torch.cuda.empty_cache()

In [ ]:
INTPUT_DIR = "" # Path to the landcover.ai dataset
CLASSES = ['background', 'building', 'woodlands', 'water', 'road']

trainSet = pd.read_csv(INTPUT_DIR + "train.txt", header=None, names=["file"])
trainSet["img"] = INTPUT_DIR + "output/" + trainSet['file'] + ".jpg"
trainSet["mask"] = INTPUT_DIR + "output/" + trainSet['file'] + "_m.png"

valSet = pd.read_csv(INTPUT_DIR + "val.txt", header=None, names=["file"])
valSet["img"] = INTPUT_DIR + "output/" + valSet['file'] + ".jpg"
valSet["mask"] = INTPUT_DIR + "output/" + valSet['file'] + "_m.png"

print("Training dataset: " + str(len(trainSet)))
print("Validation dataset: " + str(len(valSet)))

test_transform = A.Compose(
    [A.CenterCrop(height=224, width=224),])

train_transform = A.Compose(
    [A.CenterCrop(height=224, width=224),
     A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
     A.HorizontalFlip(p=0.5),
     A.VerticalFlip(p=0.5),
     A.MedianBlur(blur_limit=3, p=0.1),])

In [ ]:
class MultiClassSegDataset(Dataset):
    
    def __init__(self, df, transform=None,):
        self.df = df
        self.transform = transform
    
    def __getitem__(self, idx):
        image_name = self.df.iloc[idx, 1]
        mask_name = self.df.iloc[idx, 2]
        image = cv2.imread(image_name)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_name, cv2.IMREAD_UNCHANGED)
        image = image.astype('uint8')
        mask = mask[:,:,0]
        if(self.transform is not None):
            transformed = self.transform(image=image, mask=mask)
            image = transformed["image"]
            mask = transformed["mask"]
        image = torch.from_numpy(image)
        mask = torch.from_numpy(mask)
        image = image.permute(2, 0, 1)
        image = image.float()/255
        mask = mask.long()
        return image, mask  
        
    def __len__(self):
        return len(self.df)

In [ ]:
# Argument Parser for Mamba-based models
mamba_parser = argparse.ArgumentParser()
mamba_parser.add_argument('--root_path', type=str, default='../data/ACDC', help='Name of Experiment')
mamba_parser.add_argument('--exp', type=str, default='ACDC/Fully_Supervised', help='experiment_name')
mamba_parser.add_argument('--model', type=str, default='mambaunet', help='model_name')
mamba_parser.add_argument('--num_classes', type=int, default=5, help='output channel of network')

mamba_parser.add_argument('--cfg', type=str, default="main-codes/Mamba-UNet-main/code/configs/vmamba_tiny.yaml", help='path to config file')
mamba_parser.add_argument('--opts', default=None, nargs='+', help="Modify config options by adding 'KEY VALUE' pairs.")

mamba_parser.add_argument('--zip', action='store_true', help='use zipped dataset instead of folder dataset')
mamba_parser.add_argument('--cache-mode', type=str, default='part', choices=['no', 'full', 'part'])
mamba_parser.add_argument('--resume', help='resume from checkpoint')
mamba_parser.add_argument('--accumulation-steps', type=int)
mamba_parser.add_argument('--use-checkpoint', action='store_true')
mamba_parser.add_argument('--amp-opt-level', type=str, default='O1', choices=['O0', 'O1', 'O2'])
mamba_parser.add_argument('--tag')
mamba_parser.add_argument('--eval', action='store_true')
mamba_parser.add_argument('--throughput', action='store_true')

mamba_parser.add_argument('--max_iterations', type=int, default=25)
mamba_parser.add_argument('--batch_size', type=int, default=8)
mamba_parser.add_argument('--deterministic', type=int, default=1)
mamba_parser.add_argument('--base_lr', type=float, default=0.0001)
mamba_parser.add_argument('--patch_size', type=int, nargs=2, default=[224, 224])
mamba_parser.add_argument('--seed', type=int, default=1337)
mamba_parser.add_argument('--labeled_num', type=int, default=140)

# Step 2: Simulated args for Jupyter/IPython
mamba_simulated_args = [
    '--root_path', '../data/ACDC',
    '--exp', 'ACDC/Fully_Supervised',
    '--model', 'mambaunet',
    '--num_classes', '5',
    '--cfg', 'main-codes/Mamba-UNet-main/code/configs/vmamba_tiny.yaml',
    '--patch_size', '224', '224',
    '--batch_size', '8',
    '--max_iterations', '25',
    '--base_lr', '0.0001',
    '--seed', '1337',
    '--labeled_num', '140'
]
mamba_args = mamba_parser.parse_args(args=mamba_simulated_args)

# Step 3: Load config
mamba_config = get_config(mamba_args)

In [ ]:
# Argument Parser for Swin-based models
swin_parser = argparse.ArgumentParser()
swin_parser.add_argument('--root_path', type=str, default='../data/ACDC', help='Name of Experiment')
swin_parser.add_argument('--exp', type=str, default='ACDC/Fully_Supervised_ViT', help='experiment_name')
swin_parser.add_argument('--model', type=str, default='unet', help='model_name')
swin_parser.add_argument('--max_iterations', type=int, default=25, help='maximum epoch number to train')
swin_parser.add_argument('--batch_size', type=int, default=8, help='batch_size per gpu')
swin_parser.add_argument('--deterministic', type=int, default=1, help='whether use deterministic training')
swin_parser.add_argument('--base_lr', type=float, default=0.0001, help='segmentation network learning rate')
swin_parser.add_argument('--patch_size', type=list, default=[224, 224], help='patch size of network input')
swin_parser.add_argument('--seed', type=int,  default=1337, help='random seed')
swin_parser.add_argument('--num_classes', type=int, default=5, help='output channel of network')
swin_parser.add_argument('--cfg', type=str, default="main-codes/Mamba-UNet-main/code/configs/swin_tiny_patch4_window7_224_lite.yaml", help='path to config file', )
swin_parser.add_argument('--opts', default=None, nargs='+', help="Modify config options by adding 'KEY VALUE' pairs. ")
swin_parser.add_argument('--zip', action='store_true',
                    help='use zipped dataset instead of folder dataset')
swin_parser.add_argument('--cache-mode', type=str, default='part', choices=['no', 'full', 'part'],
                    help='no: no cache, '
                         'full: cache all data, '
                         'part: sharding the dataset into nonoverlapping pieces and only cache one piece')
swin_parser.add_argument('--resume', help='resume from checkpoint')
swin_parser.add_argument('--accumulation-steps', type=int,
                    help="gradient accumulation steps")
swin_parser.add_argument('--use-checkpoint', action='store_true',
                    help="whether to use gradient checkpointing to save memory")
swin_parser.add_argument('--amp-opt-level', type=str, default='O1', choices=['O0', 'O1', 'O2'],
                    help='mixed precision opt level, if O0, no amp is used')
swin_parser.add_argument('--tag', help='tag of experiment')
swin_parser.add_argument('--eval', action='store_true',
                    help='Perform evaluation only')
swin_parser.add_argument('--throughput', action='store_true',
                    help='Test throughput only')

# label and unlabel
swin_parser.add_argument('--labeled_bs', type=int, default=12,
                    help='labeled_batch_size per gpu')
swin_parser.add_argument('--labeled_num', type=int, default=140,
                    help='labeled data')
# costs
swin_parser.add_argument('--ema_decay', type=float,  default=0.99, help='ema_decay')
swin_parser.add_argument('--consistency_type', type=str,
                    default="mse", help='consistency_type')
swin_parser.add_argument('--consistency', type=float,
                    default=0.1, help='consistency')
swin_parser.add_argument('--consistency_rampup', type=float,
                    default=200.0, help='consistency_rampup')

# Step 2: Simulated args for Jupyter/IPython
swin_simulated_args = [
    '--root_path', '../data/ACDC',
    '--exp', 'ACDC/Fully_Supervised_ViT',
    '--model', 'unet',
    '--num_classes', '5',
    '--cfg', 'main-codes/Mamba-UNet-main/code/configs/swin_tiny_patch4_window7_224_lite.yaml',
    '--patch_size', '[224, 224]',
    '--batch_size', '8',
    '--max_iterations', '25',
    '--base_lr', '0.0001',
    '--seed', '1337',
    '--labeled_num', '140'
]
swin_args = swin_parser.parse_args(args=swin_simulated_args)

# Step 3: Load config
swin_config = get_config(swin_args)

In [ ]:
# reproducibility helper, applied to ALL architectures
# inside experimentPipe() 
def set_seed2(seed=2019):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.random.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False
    torch.use_deterministic_algorithms(True)

In [ ]:
def experimentPipe(setNum,
                   architecture,
                   saveFolder, 
                   logFile, 
                   epochs, 
                   device, 
                   modelName,
                   seed):  # per-replicate seed, applied below for every architecture

    torch.cuda.empty_cache()

    # seed before dataset/model construction so SwinUNet, SwinUNetC, and UNet
    # all start from the same RNG state for a given replicate (setNum) and are
    # reproducible across reruns.
    set_seed2(seed)
    # albumentations 2.x has its own RNG that ignores random/np.random seeds,
    # so the augmentation pipelines must be re-seeded explicitly
    train_transform.set_random_seed(seed)
    test_transform.set_random_seed(seed)

    if setNum == 1:
        trainDF = trainSet.sample(n=5000, random_state=41)
        valDF = valSet.sample(n=1600, random_state=41)
    elif setNum == 2:
        trainDF = trainSet.sample(n=5000, random_state=42)
        valDF = valSet.sample(n=1600, random_state=41)
    else:
        trainDF = trainSet.sample(n=5000, random_state=43)
        valDF = valSet.sample(n=1600, random_state=41)
    
    trainDS = MultiClassSegDataset(trainDF, transform=train_transform)
    valDS = MultiClassSegDataset(valDF, transform=test_transform)

    trainDL = DataLoader(trainDS, batch_size=8, shuffle=False, sampler=None,
                         num_workers=0, pin_memory=False, drop_last=False)
    valDL = DataLoader(valDS, batch_size=8, shuffle=False,
                       num_workers=0, pin_memory=False, drop_last=False)

    if architecture == "SwinUNet":
        model = ViT_seg(swin_config, 
                img_size=swin_args.patch_size,
                num_classes=swin_args.num_classes).to(device)
        model.load_from(swin_config)
    elif architecture == "SwinUNetC":
        model = HybridSwinUNetC(swin_config, 
            img_size=swin_args.patch_size,
            num_classes=swin_args.num_classes).to(device)
        model.load_from(swin_config)
    elif architecture == "MambaUNet":
        model = VIM_seg(mamba_config, 
                img_size=mamba_args.patch_size,
                num_classes=mamba_args.num_classes).to(device)
        model.load_from(mamba_config)
    elif architecture == "MambaUNetC":
        model = HybridMambaUNetC(mamba_config, 
                                    img_size=mamba_args.patch_size, 
                                    num_classes=mamba_args.num_classes).to(device)
        model.load_from(mamba_config)  
    else:
        model = smp.Unet(
            encoder_name = "resnet34",
            encoder_weights = "imagenet",
            decoder_channels = (768, 768, 384, 192, 96),
            in_channels = 3,
            classes = 5
            ).to(device)
        
    model = nn.DataParallel(model)
        
    print("Set number is: ", setNum, "\nThe architecture is: ", architecture)

    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001)

    criterion = smp.losses.DiceLoss(mode="multiclass", classes = 5, from_logits=True, smooth=1e-8).to(device)
    criterion.__name__ = 'dice' #Requires a name parameter
    
    acc = tm.Accuracy(task="multiclass", average="micro", num_classes=5).to(device)
    f1 = tm.F1Score(task="multiclass", average="macro", num_classes=5).to(device)
    recall = tm.Recall(task="multiclass", average="macro", num_classes=5).to(device)
    precision = tm.Precision(task="multiclass", average="macro", num_classes=5).to(device)

    eNum = []
    t_loss = []
    t_acc = []
    t_f1 = []
    t_recall = []
    t_precision = []
    
    v_loss = []
    v_acc = []
    v_f1 = []
    v_recall = []
    v_precision = []

    f1VMax = 0.0

    # Loop over epochs
    for epoch in range(1, epochs+1):
        model.train()
        running_loss = 0.0
        # Loop over training batches
        for batch_idx, (inputs, targets) in enumerate(trainDL):
            # Get data and move to device
            inputs, targets = inputs.to(device), targets.to(device)

            # Clear gradients
            optimizer.zero_grad()
            # Predict data
            outputs = model(inputs)
            # Calculate loss
            loss = criterion(outputs, targets)

            # Calculate metrics
            accT = acc(outputs, targets)
            f1T = f1(outputs, targets)
            recallT = recall(outputs, targets)
            precisionT = precision(outputs, targets)
            
            # Backpropagate
            loss.backward()

            # Update parameters
            optimizer.step()

            #Update running with batch results
            running_loss += loss.item()

        # Accumulate loss and metrics at end of training epoch
        epoch_loss = running_loss/len(trainDL)
        accT = acc.compute()
        f1T = f1.compute()
        recallT = recall.compute()
        precisionT = precision.compute()

        # Print Losses and metrics at end of each training epoch
        print(f'Epoch: {epoch}, Training Loss: {epoch_loss:.4f}, Training Accuracy: {accT:.4f}, Training F1: {f1T:.4f}, Training Recall: {recallT:.4f}, Training Precision: {precisionT:.4f}')   

        # Append results
        eNum.append(epoch)
        t_loss.append(epoch_loss)
        t_acc.append(accT.detach().cpu().numpy())
        t_f1.append(f1T.detach().cpu().numpy())
        t_recall.append(recallT.detach().cpu().numpy())
        t_precision.append(precisionT.detach().cpu().numpy())

        # Reset metrics
        acc.reset()
        f1.reset()
        recall.reset()
        precision.reset()

        model.eval()
        
        # loop over validation batches
        with torch.no_grad():
            #Initialize running validation loss
            running_loss_v = 0.0
            for batch_idx, (inputsV, targetsV) in enumerate(valDL):
                # Get data and move to device
                inputsV, targetsV = inputsV.to(device), targetsV.to(device)

                # Predict data
                outputsV = model(inputsV)
                # Calculate validation loss
                loss_v = criterion(outputsV, targetsV)
                
                #Update running with batch results
                running_loss_v += loss_v.item()

                # Calculate metrics
                accV = acc(outputsV, targetsV)
                f1V = f1(outputsV, targetsV)
                recallV = recall(outputsV, targetsV)
                precisionV = precision(outputsV, targetsV)
                
        #Accumulate loss and metrics at end of validation epoch
        epoch_loss_v = running_loss_v/len(valDL)
        accV = acc.compute()
        f1V = f1.compute()
        recallV = recall.compute()
        precisionV = precision.compute()

        # Print validation loss and metrics
        print(f'Validation Loss: {epoch_loss_v:.4f}, Validation Accuracy: {accV:.4f}, Validation F1: {f1V:.4f}, Validation Recall: {recallV:.4f}, Validation Precision: {precisionV:.4f}')
        
        # Append results
        v_loss.append(epoch_loss_v)
        v_acc.append(accV.detach().cpu().numpy())
        v_f1.append(f1V.detach().cpu().numpy())
        v_recall.append(recallV.detach().cpu().numpy())
        v_precision.append(precisionV.detach().cpu().numpy())

        # Reset metrics
        acc.reset()
        f1.reset()
        recall.reset()
        precision.reset()

        # Save model if validation F1-score improves
        f1V2 = f1V.detach().cpu().numpy()
        if f1V2 > f1VMax:
            f1VMax = f1V2
            torch.save(model.state_dict(), saveFolder + modelName)
            print(f'Model saved for epoch {epoch}.')


    SeNum = pd.Series(eNum, name="epoch")
    St_loss = pd.Series(t_loss, name="training_loss")
    St_acc = pd.Series(t_acc, name="training_accuracy")
    St_f1 = pd.Series(t_f1, name="training_f1")
    St_recall = pd.Series(t_recall, name="train_recall")
    St_precision = pd.Series(t_precision, name="train_precision")
    Sv_loss = pd.Series(v_loss, name="val_loss")
    Sv_acc = pd.Series(v_acc, name="val_accuracy")
    Sv_f1 = pd.Series(v_f1, name="val_f1")
    Sv_recall = pd.Series(v_recall, name="val_recall")
    Sv_precision = pd.Series(v_precision, name="val_precision")
    resultsDF = pd.concat([SeNum, St_loss, St_acc, St_f1, St_recall, St_precision, 
                            Sv_loss, Sv_acc, Sv_f1, Sv_recall, Sv_precision], 
                            axis=1)
    resultsDF.to_csv(saveFolder + logFile)
    
    torch.cuda.empty_cache()

In [ ]:
OUTPUT_DIR = "" # Path to save trained models
num = [1, 2, 3]
arch = ["SwinUNet", "SwinUNetC", "MambaUNet", "MambaUNetC","UNet"]

for n in num:
    seed_n = swin_args.seed + n #(1338, 1339, 1340)
    for a in arch:
        csvfile = a + "_S" + str(n) + ".csv"
        modelfile = a + "_S" + str(n) + ".pt"
        experimentPipe(setNum = n,
                       architecture = a,
                       saveFolder = OUTPUT_DIR, 
                       logFile = csvfile, 
                       epochs = 25, 
                       device = "cuda", 
                       modelName = modelfile,
                       seed = seed_n)